# ATLAS Foundation Model - Prototype Semplificato

Questo notebook implementa una versione proof-of-concept del progetto descritto nel piano.
Features implementate:
- Generazione di dati sintetici semplificati (tracce di particelle)
- Tokenizzazione e embedding
- Architettura Perceiver gerarchica miniaturizzata
- Obiettivi di pre-training (masked modeling)

# ## 1. Installazioni e Import

In [1]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math
from typing import Dict, List, Optional, Tuple
import warnings
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd
from dataclasses import dataclass
import torch.optim as optim
from torch.utils.data import random_split
import os

In [2]:
@dataclass
class AtlasTrack:
    pt: float
    eta: float
    phi: float
    e: float
    d0: float
    z0: float
    charge: int
    track_id: str
    vertex_id: int
    is_pileup: bool


class RealisticAtlasGenerator:

    def __init__(
        self,
        avg_tracks: int = 120,
        avg_pileup_vertices: int = 20,
        eta_range: Tuple[float, float] = (-2.5, 2.5),
        pt_min: float = 0.5,
        random_seed: Optional[int] = None,
    ):

        self.avg_tracks = avg_tracks
        self.avg_pileup_vertices = avg_pileup_vertices
        self.eta_range = eta_range
        self.pt_min = pt_min

        if random_seed:
            np.random.seed(random_seed)

    # -------------------------
    # Distribuzioni pT
    # -------------------------

    def sample_pt_primary(self):
        """Distribuzione pT più dura per tracce primarie"""
        alpha = 3
        u = np.random.random()
        return self.pt_min * (1 - u) ** (-1 / alpha)

    def sample_pt_pileup(self):
        """Distribuzione pT più soft per pileup"""
        alpha = 5
        u = np.random.random()
        return self.pt_min * (1 - u) ** (-1 / alpha)

    # -------------------------
    # Energia
    # -------------------------

    def compute_energy(self, pt, eta):
        mass = 0.14
        p = pt * np.cosh(eta)
        return np.sqrt(p**2 + mass**2)

    # -------------------------
    # Generazione traccia
    # -------------------------

    def generate_track(
        self,
        track_id,
        vertex_z,
        vertex_id,
        is_pileup,
        jet_center=None,
    ):

        # pT diverso
        if is_pileup:
            pt = self.sample_pt_pileup()
        else:
            pt = self.sample_pt_primary()

        # distribuzione angolare
        if jet_center is not None:
            eta = np.random.normal(jet_center[0], 0.1)
            phi = np.random.normal(jet_center[1], 0.1)
        else:
            eta = np.random.uniform(*self.eta_range)
            phi = np.random.uniform(0, 2 * np.pi)

        # energia
        e = self.compute_energy(pt, eta)

        # parametri impatto
        if is_pileup:
            d0 = np.random.normal(0, 0.5)
        else:
            d0 = np.random.normal(0, 0.05)

        z0 = vertex_z + np.random.normal(0, 1)

        charge = np.random.choice([-1, 1])

        return AtlasTrack(
            pt=pt,
            eta=eta,
            phi=phi,
            e=e,
            d0=d0,
            z0=z0,
            charge=charge,
            track_id=track_id,
            vertex_id=vertex_id,
            is_pileup=is_pileup,
        )

    # -------------------------
    # Generazione evento
    # -------------------------

    def generate_event(self, event_id):

        tracks = []

        # -------------------------
        # Primary vertex
        # -------------------------

        primary_z = np.random.normal(0, 50)

        n_primary_tracks = np.random.poisson(self.avg_tracks * 0.5)

        # genera jet
        n_jets = np.random.randint(2, 5)
        jets = [
            (
                np.random.uniform(*self.eta_range),
                np.random.uniform(0, 2 * np.pi),
            )
            for _ in range(n_jets)
        ]

        for i in range(n_primary_tracks):

            # metà delle tracce nei jet
            if np.random.random() < 0.6:
                jet = jets[np.random.randint(len(jets))]
            else:
                jet = None

            track = self.generate_track(
                track_id=f"{event_id}_P_{i}",
                vertex_z=primary_z,
                vertex_id=0,
                is_pileup=False,
                jet_center=jet,
            )

            tracks.append(track)

        # -------------------------
        # Pileup vertices
        # -------------------------

        n_pileup_vertices = np.random.poisson(self.avg_pileup_vertices)

        track_counter = 0

        for v in range(n_pileup_vertices):

            z_vertex = np.random.normal(0, 50)

            n_tracks = np.random.poisson(5)

            for i in range(n_tracks):

                track = self.generate_track(
                    track_id=f"{event_id}_PU_{track_counter}",
                    vertex_z=z_vertex,
                    vertex_id=v + 1,
                    is_pileup=True,
                    jet_center=None,
                )

                tracks.append(track)
                track_counter += 1

        return self.tracks_to_dataframe(event_id, tracks)

    # -------------------------
    # Conversione dataframe
    # -------------------------

    def tracks_to_dataframe(self, event_id, tracks: List[AtlasTrack]):

        df = pd.DataFrame([
            {
                "event_id": event_id,
                "track_id": t.track_id,
                "vertex_id": t.vertex_id,
                "pt": t.pt,
                "eta": t.eta,
                "phi": t.phi,
                "energy": t.e,
                "d0": t.d0,
                "z0": t.z0,
                "charge": t.charge,
                "is_pileup": t.is_pileup,
                "log_pt": np.log(t.pt),
                "px": t.pt * np.cos(t.phi),
                "py": t.pt * np.sin(t.phi),
                "pz": t.pt * np.sinh(t.eta),
                "p": t.pt * np.cosh(t.eta),
            }
            for t in tracks
        ])

        return df

    # -------------------------
    # Dataset
    # -------------------------

    def generate_dataset(self, n_events):

        events = []

        for i in range(n_events):
            events.append(self.generate_event(i))

        return pd.concat(events, ignore_index=True)


# Salvataggio in formato parquet per uso futuro   
def save_dataset(dataset: pd.DataFrame, filename: str):
    """Salva il dataset in formato parquet (efficiente)"""
    dataset.to_parquet(filename, index=False)
    print(f"Dataset salvato in {filename}")

# -------------------------
# Demo
# -------------------------

if __name__ == "__main__":

    generator = RealisticAtlasGenerator(
        avg_tracks=120,
        avg_pileup_vertices=20,
        random_seed=42,
    )

    dataset = generator.generate_dataset(1000)

    # Esempio di salvataggio 
    save_dataset(dataset, "atlas_tracks_sample.parquet")

    print("Dataset generato:")
    print(dataset.head())

    print("\nStatistiche:")
    print("Tracce totali:", len(dataset))
    print("Eventi:", dataset.event_id.nunique())
    print("Frazione pileup:", dataset.is_pileup.mean())

Dataset salvato in atlas_tracks_sample.parquet
Dataset generato:
   event_id track_id  vertex_id        pt       eta       phi    energy  \
0         0    0_P_0          0  0.500130  2.461058  3.879751  2.954721   
1         0    0_P_1          0  1.682509 -2.260721  4.511348  8.156557   
2         0    0_P_2          0  0.962792 -1.636968  4.037141  2.571642   
3         0    0_P_3          0  0.548144 -2.106000  4.631336  2.289231   
4         0    0_P_4          0  0.799457 -2.180386  4.398658  3.585387   

         d0         z0  charge  is_pileup    log_pt        px        py  \
0 -0.006913  24.857929      -1      False -0.692887 -0.369951 -0.336551   
1 -0.008193  23.359378       1      False  0.520286 -0.335980 -1.648622   
2  0.095639  22.808988       1      False -0.037918 -0.601833 -0.751509   
3  0.079043  24.467537      -1      False -0.601218 -0.044380 -0.546344   
4 -0.017656  24.374242       1      False -0.223823 -0.246720 -0.760434   

         pz         p  
0  2.9087

In [3]:
class AtlasTrackDataset(Dataset):
    def __init__(self, dataframe, stats):
        """
        dataframe: il DF con tutte le tracce, con colonna 'event_id'
        stats: dizionario con 'mean' e 'std' per ogni feature
        """
        self.dataframe = dataframe
        self.event_ids = dataframe['event_id'].unique()
        self.stats = stats
        
        # Features per i token PF (quelle che andranno nel modello)
        self.pf_features = [
            'pt', 'log_pt', 'eta', 'phi_x', 'phi_y', 
            'energy', 'd0', 'z0', 'charge'
        ]
        
        print(f"Dataset inizializzato con {len(self.event_ids)} eventi")
        print(f"Features PF: {self.pf_features}")
        
    def __len__(self):
        return len(self.event_ids)
    
    def __getitem__(self, idx):
        event_id = self.event_ids[idx]
        event_data = self.dataframe[self.dataframe['event_id'] == event_id].copy()
        
        # --- Calcola feature derivate ---
        # Feature angolari (necessarie per positional encoding)
        event_data['phi_x'] = np.cos(event_data['phi'])
        event_data['phi_y'] = np.sin(event_data['phi'])
        
        # --- 1. Token PF (con normalizzazione) ---
        pf_tokens_list = []
        for feat in self.pf_features:
            if feat in ['charge', 'is_pileup']:  # Non normalizzare variabili categoriche
                values = event_data[feat].values.astype(np.float32)
            else:
                # Normalizza
                mean = self.stats['mean'][feat]
                std = self.stats['std'][feat]
                values = (event_data[feat].values - mean) / std
            pf_tokens_list.append(values)
        
        # Stack features: [n_tracce, n_features]
        pf_tokens = np.stack(pf_tokens_list, axis=1).astype(np.float32)
        
        # --- 2. Token Globali ---
        ht = event_data['pt'].sum()
        n_tracks = len(event_data)
        
        # Normalizza HT
        ht_norm = (ht - self.stats['mean']['HT']) / self.stats['std']['HT']
        global_tokens = np.array([ht_norm, n_tracks], dtype=np.float32)
        
        # --- 3. Eta e Phi (per attenzione locale) ---
        eta = event_data['eta'].values.astype(np.float32)
        phi = event_data['phi'].values.astype(np.float32)
        
        # --- 4. Labels per pre-addestramento ---
        is_pileup_labels = event_data['is_pileup'].values.astype(np.float32)
        
        return {
            'pf_tokens': torch.from_numpy(pf_tokens),
            'global_tokens': torch.from_numpy(global_tokens),
            'eta': torch.from_numpy(eta),
            'phi': torch.from_numpy(phi),
            'is_pileup_labels': torch.from_numpy(is_pileup_labels),
            'n_tracks': n_tracks,
            'event_id': event_id
        }

In [4]:
class PositionalEncoding(nn.Module):
    """Encoding posizionale per η-φ (geometria cilindrica)"""
    def __init__(self, d_model, max_eta_range=5.0, n_phi_bins=64):
        super().__init__()
        self.d_model = d_model
        self.max_eta_range = max_eta_range
        
        # Dividi d_model in parti uguali per eta e phi
        self.eta_dim = d_model // 2
        self.phi_dim = d_model - self.eta_dim  # gestisce il caso in cui d_model sia dispari
        
        self.eta_lin = nn.Linear(1, self.eta_dim)
        self.phi_lin = nn.Linear(2, self.phi_dim)
        
    def forward(self, eta, phi):
        """
        eta: [batch, n_tokens] o [n_tokens]
        phi: [batch, n_tokens] o [n_tokens]
        """
        # Normalizza eta in [-1, 1] circa
        eta_norm = eta / self.max_eta_range
        
        # Trasforma phi in coordinate circolari
        phi_cos = torch.cos(phi)
        phi_sin = torch.sin(phi)
        phi_stack = torch.stack([phi_cos, phi_sin], dim=-1)
        
        # Proietta
        eta_emb = self.eta_lin(eta_norm.unsqueeze(-1))  # [batch, n, eta_dim]
        phi_emb = self.phi_lin(phi_stack)  # [batch, n, phi_dim]
        
        # Concatena lungo l'ultima dimensione
        return torch.cat([eta_emb, phi_emb], dim=-1)  # [batch, n, d_model]


class ModalityEncoder(nn.Module):
    """Encoder per ogni modalità (PF, Globali, ecc.)"""
    def __init__(self, input_dim, d_model, hidden_dim=128):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model)
        )
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        return self.layer_norm(self.input_proj(x))


class SimpleLocalAttentionBlock(nn.Module):
    """Versione semplificata che usa una finestra fissa senza maschera complessa"""
    def __init__(self, d_model, n_heads, window_size=0.5, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.window_size = window_size
        
        # Attenzione standard
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model)
        )
        
    def forward(self, x, eta, phi):
        """
        x: [batch, n_tokens, d_model]
        eta, phi: [batch, n_tokens] (non usati in questa versione semplificata)
        """
        # Self-attention standard (senza maschera)
        attn_out, _ = self.attention(x, x, x)
        
        # Residual
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        
        return x


class PerceiverCore(nn.Module):
    """Core Perceiver per fusione globale"""
    def __init__(self, d_model, n_latents=256, n_layers=6, n_heads=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_latents = n_latents
        
        # Latent array apprendibile
        self.latents = nn.Parameter(torch.randn(1, n_latents, d_model) * 0.02)
        
        # Layer per cross-attention e self-attention
        self.cross_attn_layers = nn.ModuleList()
        self.self_attn_layers = nn.ModuleList()
        
        for _ in range(n_layers):
            # Cross-attention: latents -> tokens
            self.cross_attn_layers.append(
                nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            )
            # Self-attention tra latents
            self.self_attn_layers.append(
                nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            )
        
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers * 2)])
        self.ffns = nn.ModuleList([nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model)
        ) for _ in range(n_layers * 2)])
        
    def forward(self, tokens, mask=None):
        """
        tokens: [batch, n_tokens, d_model]
        mask: maschera per padding [batch, n_tokens] (True dove è padding)
        """
        batch_size = tokens.shape[0]
        
        # Espandi latents per il batch
        latents = self.latents.expand(batch_size, -1, -1)
        
        # Crea chiave/valore mask per cross-attention
        key_padding_mask = mask if mask is not None else None
        
        for i in range(len(self.cross_attn_layers)):
            # Cross-attention: latents -> tokens
            latents_norm = self.norms[i*2](latents)
            cross_out, _ = self.cross_attn_layers[i](
                latents_norm, tokens, tokens,
                key_padding_mask=key_padding_mask
            )
            latents = latents + cross_out
            
            # FFN dopo cross
            latents_norm = self.norms[i*2](latents)
            ffn_out = self.ffns[i*2](latents_norm)
            latents = latents + ffn_out
            
            # Self-attention tra latents
            latents_norm = self.norms[i*2 + 1](latents)
            self_out, _ = self.self_attn_layers[i](
                latents_norm, latents_norm, latents_norm
            )
            latents = latents + self_out
            
            # FFN dopo self
            latents_norm = self.norms[i*2 + 1](latents)
            ffn_out = self.ffns[i*2 + 1](latents_norm)
            latents = latents + ffn_out
            
        return latents


class HierarchicalPerceiver(nn.Module):
    """
    Modello completo: Perceiver Gerarchico per ATLAS
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        d_model = config['d_model']
        
        # 1. Positional encoding per geometria
        self.pos_encoding = PositionalEncoding(
            d_model, 
            max_eta_range=config.get('max_eta', 5.0),
            n_phi_bins=config.get('n_phi_bins', 64)
        )
        
        # 2. Encoder per modalità
        self.pf_encoder = ModalityEncoder(
            config['pf_input_dim'], d_model, config.get('hidden_dim', 128)
        )
        self.global_encoder = ModalityEncoder(
            config['global_input_dim'], d_model, config.get('hidden_dim', 64)
        )
        
        # Opzionale: encoder per alto livello (quando disponibili)
        if config.get('use_high_level', False):
            self.high_level_encoder = ModalityEncoder(
                config['high_level_input_dim'], d_model, config.get('hidden_dim', 128)
            )
        
        # 3. Local mixer per PF - usa la versione SEMPLIFICATA
        self.local_mixers = nn.ModuleList([
            SimpleLocalAttentionBlock(  # Cambiato qui!
                d_model, 
                config.get('n_heads', 8),
                window_size=config.get('local_window', 0.5),
                dropout=config.get('dropout', 0.1)
            )
            for _ in range(config.get('n_local_layers', 2))
        ])
        
        # 4. Perceiver core per fusione globale
        self.perceiver_core = PerceiverCore(
            d_model,
            n_latents=config.get('n_latents', 256),
            n_layers=config.get('n_perceiver_layers', 6),
            n_heads=config.get('n_heads', 8),
            dropout=config.get('dropout', 0.1)
        )
        
        # 5. Teste per pre-addestramento
        # Testa A: Masked modeling su PF
        self.masked_pf_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, config['pf_input_dim'])
        )
        
        # Testa B: PU-aware (predici IsPU)
        self.pu_head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Testa C: Jet-constituent (placeholder per ora)
        # ...
        
        self._init_weights()
        
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
                    
    def forward(self, batch, task='pretrain'):
        """
        batch: dizionario dal DataLoader
        task: 'pretrain', 'masked', 'pu_only', ecc.
        """
        pf_tokens_raw = batch['pf_tokens']  # [batch, n_pf, pf_dim]
        global_tokens_raw = batch['global_tokens']  # [batch, global_dim]
        eta = batch['eta']  # [batch, n_pf]
        phi = batch['phi']  # [batch, n_pf]
        mask = batch.get('mask', None)  # [batch, n_pf], True per padding
        
        batch_size, n_pf, _ = pf_tokens_raw.shape
        
        # 1. Encode PF tokens
        pf_embedded = self.pf_encoder(pf_tokens_raw)  # [batch, n_pf, d_model]
        
        # 2. Aggiungi positional encoding
        pos_emb = self.pos_encoding(eta, phi)  # [batch, n_pf, d_model]
        pf_embedded = pf_embedded + pos_emb
        
        # 3. Local mixing su PF (attenzione in η-φ)
        for mixer in self.local_mixers:
            pf_embedded = mixer(pf_embedded, eta, phi)
        
        # 4. Encode global tokens
        global_embedded = self.global_encoder(global_tokens_raw)  # [batch, d_model]
        global_embedded = global_embedded.unsqueeze(1)  # [batch, 1, d_model]
        
        # 5. Concatena tutti i token
        all_tokens = [pf_embedded, global_embedded]
        
        # Aggiungi high-level se presenti
        if 'high_level_tokens' in batch and batch['high_level_tokens'] is not None:
            high_level_embedded = self.high_level_encoder(batch['high_level_tokens'])
            all_tokens.append(high_level_embedded)
            
        all_tokens = torch.cat(all_tokens, dim=1)  # [batch, n_total, d_model]
        
        # Crea maschera combinata
        if mask is not None:
            # Aggiungi False per global token (non è padding)
            global_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=mask.device)
            all_mask = torch.cat([mask, global_mask], dim=1)
        else:
            all_mask = None
        
        # 6. Perceiver core (fusione globale)
        latents = self.perceiver_core(all_tokens, all_mask)
        
        outputs = {}
        
        if task in ['pretrain', 'masked']:
            # Testa A: Ricostruisci PF originali (solo sui token non mascherati)
            # In una versione completa, qui applicheresti una maschera casuale
            pf_output = self.masked_pf_head(pf_embedded)  # [batch, n_pf, pf_dim]
            outputs['pf_reconstruction'] = pf_output
            
        if task in ['pretrain', 'pu_only']:
            # Testa B: Predici is_pileup per ogni PF
            pu_logits = self.pu_head(pf_embedded).squeeze(-1)  # [batch, n_pf]
            outputs['pu_predictions'] = pu_logits
            
        # Latents possono essere usati per altri compiti
        outputs['latents'] = latents
        outputs['pf_embeddings'] = pf_embedded
        
        return outputs

In [5]:
def get_model_config():
    """Configurazione del modello basata sul tuo piano"""
    config = {
        'd_model': 256,  # Dimensione embedding
        'pf_input_dim': 9,  # pt, log_pt, eta, phi_x, phi_y, energy, d0, z0, charge
        'global_input_dim': 2,  # HT, n_tracks (per ora)
        'hidden_dim': 128,
        'n_heads': 8,
        'n_local_layers': 2,
        'n_perceiver_layers': 4,
        'n_latents': 128,  # Numero di latents per comprimere l'evento
        'local_window': 0.5,  # Finestra in η-φ per attenzione locale
        'dropout': 0.1,
        'max_eta': 2.5,
        'use_high_level': False,  # Non abbiamo ancora oggetti di alto livello
    }
    return config

In [6]:
class Trainer:
    def __init__(self, model, device, config):
        self.model = model
        self.device = device
        self.config = config
        
        # Optimizer
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=config.get('lr', 1e-4),
            weight_decay=config.get('weight_decay', 0.01)
        )
        
        # Scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, 
            T_max=config.get('n_epochs', 100)
        )
        
        # Loss functions
        self.mse_loss = nn.MSELoss()
        self.bce_loss = nn.BCELoss()
        
        # Tracking
        self.train_losses = []
        self.val_losses = []
        
    def compute_loss(self, outputs, batch, task_weights):
        """Calcola loss combinata per pre-addestramento"""
        total_loss = 0
        losses = {}
        
        # Loss A: Masked modeling (ricostruzione PF)
        if 'pf_reconstruction' in outputs and task_weights.get('masked', 0) > 0:
            pf_recon = outputs['pf_reconstruction']
            pf_target = batch['pf_tokens']  # [batch, n_pf, pf_dim]
            mask = ~batch['mask'] if 'mask' in batch else torch.ones_like(pf_target[..., 0], dtype=torch.bool)
            
            # Calcola MSE solo sui token validi
            recon_loss = self.mse_loss(
                pf_recon[mask], 
                pf_target[mask]
            )
            losses['masked'] = recon_loss
            total_loss += task_weights['masked'] * recon_loss
            
        # Loss B: PU-aware (predici is_pileup)
        if 'pu_predictions' in outputs and task_weights.get('pu', 0) > 0:
            pu_pred = outputs['pu_predictions']
            pu_target = batch['is_pileup_labels']  # [batch, n_pf]
            mask = ~batch['mask'] if 'mask' in batch else torch.ones_like(pu_target, dtype=torch.bool)
            
            pu_loss = self.bce_loss(
                pu_pred[mask],
                pu_target[mask]
            )
            losses['pu'] = pu_loss
            total_loss += task_weights['pu'] * pu_loss
            
        return total_loss, losses
    
    def train_epoch(self, train_loader, task_weights):
        self.model.train()
        total_loss = 0
        all_losses = {}
        
        pbar = tqdm(train_loader, desc='Training')
        for batch in pbar:
            # Sposta su GPU
            batch = {k: v.to(self.device) if torch.is_tensor(v) else v 
                     for k, v in batch.items()}
            
            self.optimizer.zero_grad()
            
            # Forward
            outputs = self.model(batch, task='pretrain')
            
            # Compute loss
            loss, losses = self.compute_loss(outputs, batch, task_weights)
            
            # Backward
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 
                                           self.config.get('max_grad_norm', 1.0))
            
            self.optimizer.step()
            
            # Track
            total_loss += loss.item()
            for k, v in losses.items():
                all_losses[k] = all_losses.get(k, 0) + v.item()
            
            # Update progress bar
            pbar.set_postfix({'loss': loss.item()})
            
        return total_loss / len(train_loader), {k: v/len(train_loader) for k, v in all_losses.items()}
    
    def validate(self, val_loader, task_weights):
        self.model.eval()
        total_loss = 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc='Validating'):
                batch = {k: v.to(self.device) if torch.is_tensor(v) else v 
                         for k, v in batch.items()}
                
                outputs = self.model(batch, task='pretrain')
                loss, _ = self.compute_loss(outputs, batch, task_weights)
                total_loss += loss.item()
                
        return total_loss / len(val_loader)
    
    def train(self, train_loader, val_loader, n_epochs, task_weights, save_path='checkpoints'):
        os.makedirs(save_path, exist_ok=True)
        
        for epoch in range(n_epochs):
            print(f"\nEpoch {epoch+1}/{n_epochs}")
            
            # Train
            train_loss, train_detail = self.train_epoch(train_loader, task_weights)
            self.train_losses.append(train_loss)
            
            # Validate
            val_loss = self.validate(val_loader, task_weights)
            self.val_losses.append(val_loss)
            
            # Scheduler step
            self.scheduler.step()
            
            print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            print(f"Detail: {train_detail}")
            
            # Save checkpoint
            if (epoch + 1) % 10 == 0:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                }, f"{save_path}/checkpoint_epoch_{epoch+1}.pt")
                
        return self.train_losses, self.val_losses

In [7]:
def compute_dataset_stats(df):
    """Calcola media e std per ogni feature"""
    stats = {'mean': {}, 'std': {}}
    
    # Features per PF
    pf_features = ['pt', 'log_pt', 'eta', 'energy', 'd0', 'z0']
    for feat in pf_features:
        stats['mean'][feat] = df[feat].mean()
        stats['std'][feat] = df[feat].std()
    
    # Features derivate (phi_x, phi_y) - media dovrebbe essere ~0
    stats['mean']['phi_x'] = 0
    stats['std']['phi_x'] = 0.707  # circa 1/√2
    stats['mean']['phi_y'] = 0
    stats['std']['phi_y'] = 0.707
    
    # Categoriche (charge, is_pileup) - non normalizzare
    stats['mean']['charge'] = 0
    stats['std']['charge'] = 1
    stats['mean']['is_pileup'] = 0
    stats['std']['is_pileup'] = 1
    
    # Calcola HT per evento
    ht_per_event = df.groupby('event_id')['pt'].sum()
    stats['mean']['HT'] = ht_per_event.mean()
    stats['std']['HT'] = ht_per_event.std()
    
    return stats

def collate_fn(batch):
    """
    Raggruppa più eventi in un batch con padding
    """
    # Estrai liste
    pf_tokens_list = [item['pf_tokens'] for item in batch]
    global_tokens_list = [item['global_tokens'] for item in batch]
    eta_list = [item['eta'] for item in batch]
    phi_list = [item['phi'] for item in batch]
    pu_labels_list = [item['is_pileup_labels'] for item in batch]
    
    # Trova il numero massimo di tracce in questo batch
    max_tracks = max(t.shape[0] for t in pf_tokens_list)
    feat_dim = pf_tokens_list[0].shape[1]
    
    # Prepara tensori con padding
    batch_size = len(batch)
    padded_pf = torch.zeros(batch_size, max_tracks, feat_dim)
    padded_eta = torch.zeros(batch_size, max_tracks)
    padded_phi = torch.zeros(batch_size, max_tracks)
    padded_pu = torch.zeros(batch_size, max_tracks)
    mask = torch.ones(batch_size, max_tracks, dtype=torch.bool)  # True = padding
    
    for i, (pf, eta, phi, pu) in enumerate(zip(pf_tokens_list, eta_list, phi_list, pu_labels_list)):
        n_tracks = pf.shape[0]
        padded_pf[i, :n_tracks] = pf
        padded_eta[i, :n_tracks] = eta
        padded_phi[i, :n_tracks] = phi
        padded_pu[i, :n_tracks] = pu
        mask[i, :n_tracks] = False  # False = token reale
    
    # Stack global tokens (non hanno bisogno di padding)
    global_tokens = torch.stack(global_tokens_list)  # [batch, global_dim]
    
    return {
        'pf_tokens': padded_pf,
        'global_tokens': global_tokens,
        'eta': padded_eta,
        'phi': padded_phi,
        'mask': mask,
        'is_pileup_labels': padded_pu,
    }

In [8]:
def visualize_event(model, dataset, event_idx, device):
    """Visualizza le predizioni del modello su un evento specifico"""
    model.eval()
    
    # Prendi un evento dal dataset
    event_data = dataset[event_idx]
    
    # Prepara batch (singolo evento)
    batch = collate_fn([event_data])
    batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
    
    with torch.no_grad():
        outputs = model(batch, task='pretrain')
    
    # Calcola le loss per questo evento (opzionale)
    # Dobbiamo definire le loss anche qui
    mse_loss = nn.MSELoss()
    bce_loss = nn.BCELoss()
    
    mask = batch['mask']
    valid_mask = ~mask
    
    losses = {}
    
    # Calcola loss ricostruzione se disponibile
    if 'pf_reconstruction' in outputs:
        pf_recon = outputs['pf_reconstruction']
        pf_target = batch['pf_tokens']
        recon_loss = mse_loss(
            pf_recon[valid_mask], 
            pf_target[valid_mask]
        )
        losses['masked'] = recon_loss.item()
    
    # Calcola loss pileup se disponibile
    if 'pu_predictions' in outputs:
        pu_pred = outputs['pu_predictions']
        pu_target = batch['is_pileup_labels']
        pu_loss = bce_loss(
            pu_pred[valid_mask],
            pu_target[valid_mask]
        )
        losses['pu'] = pu_loss.item()
    
    # Estrai dati per visualizzazione
    pf_tokens = batch['pf_tokens'][0].cpu().numpy()
    mask_np = batch['mask'][0].cpu().numpy()
    valid_idx = ~mask_np
    
    eta = batch['eta'][0].cpu().numpy()[valid_idx]
    phi = batch['phi'][0].cpu().numpy()[valid_idx]
    
    # Crea figura con subplot
    fig = plt.figure(figsize=(20, 12))
    
    # 1. Mappa η-φ delle tracce
    ax1 = plt.subplot(2, 3, 1)
    pu_true = batch['is_pileup_labels'][0].cpu().numpy()[valid_idx]
    
    ax1.scatter(eta[pu_true == 0], phi[pu_true == 0], 
                c='blue', label='Non-Pileup', alpha=0.6, s=20)
    ax1.scatter(eta[pu_true == 1], phi[pu_true == 1], 
                c='red', label='Pileup', alpha=0.6, s=20)
    ax1.set_xlabel('η')
    ax1.set_ylabel('φ')
    ax1.set_title('Tracce nell\'evento (True Labels)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Distribuzione pT
    ax2 = plt.subplot(2, 3, 2)
    pt_true = pf_tokens[valid_idx, 0]  # Assumendo pt sia il primo indice
    if 'pf_reconstruction' in outputs:
        pf_recon = outputs['pf_reconstruction'][0].cpu().numpy()
        pt_pred = pf_recon[valid_idx, 0]
        
        ax2.scatter(pt_true, pt_pred, alpha=0.5, s=10)
        
        # Calcola linea di regressione per meglio visualizzare
        ax2.plot([pt_true.min(), pt_true.max()], 
                [pt_true.min(), pt_true.max()], 'k--', alpha=0.5, label='Ideal')
        
        ax2.set_xlabel('True pT')
        ax2.set_ylabel('Predicted pT')
        ax2.set_title(f'pT Ricostruzione (corr={np.corrcoef(pt_true, pt_pred)[0,1]:.3f})')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    # 3. Predizioni pileup
    ax3 = plt.subplot(2, 3, 3)
    if 'pu_predictions' in outputs:
        pu_pred = outputs['pu_predictions'][0].cpu().numpy()[valid_idx]
        
        ax3.hist(pu_pred[pu_true == 0], range=[0,1], bins=50, alpha=0.5, label='Non-Pileup', color='blue', density=True)
        ax3.hist(pu_pred[pu_true == 1], range=[0,1], bins=50, alpha=0.5, label='Pileup', color='red', density=True)
        ax3.set_xlabel('Predicted Pileup Probability')
        ax3.set_ylabel('Density')
        ax3.set_title(f'Pileup Predictions (AUC={roc_auc_score(pu_true, pu_pred):.3f})' 
                     if 'roc_auc_score' in dir() else 'Pileup Predictions')
        ax3.legend()
    
    # 4. Errori di ricostruzione per feature
    ax4 = plt.subplot(2, 3, 4)
    if 'pf_reconstruction' in outputs:
        n_features = min(5, pf_tokens.shape[1])
        errors = []
        feature_names = ['pt', 'log_pt', 'eta', 'energy', 'd0', 'z0', 'charge', 'is_pileup'][:n_features]
        
        for i in range(n_features):
            err = np.abs(pf_recon[valid_idx, i] - pf_tokens[valid_idx, i])
            errors.append(err.mean())
        
        bars = ax4.bar(feature_names, errors)
        ax4.set_ylabel('Mean Absolute Error')
        ax4.set_title('Ricostruzione Errori per Feature')
        ax4.tick_params(axis='x', rotation=45)
        
    
    # 5. Distribuzione errori
    ax5 = plt.subplot(2, 3, 5)
    if 'pf_reconstruction' in outputs:
        all_errors = []
        for i in range(n_features):
            err = np.abs(pf_recon[valid_idx, i] - pf_tokens[valid_idx, i])
            all_errors.extend(err)
        
        ax5.hist(all_errors, bins=30, alpha=0.7)
        ax5.set_xlabel('Ricostruzione Error')
        ax5.set_ylabel('Count')
        ax5.set_title(f'Distribuzione Errori (media={np.mean(all_errors):.3f})')
        ax5.axvline(np.mean(all_errors), color='r', linestyle='--', label='Media')
        ax5.legend()
    
    # 6. Info evento e loss
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('off')
    
    # Calcola alcune metriche aggiuntive
    pu_accuracy = 0
    pu_auc = 0
    
    if 'pu_predictions' in outputs:
        pu_pred_binary = (pu_pred > 0.5).astype(int)
        pu_accuracy = accuracy_score(pu_true, pu_pred_binary)
        try:
            pu_auc = roc_auc_score(pu_true, pu_pred)
        except:
            pu_auc = 0
    
    info_text = f"""
    {'='*30}
    Evento {event_idx}
    {'='*30}
    
    Statistiche Evento:
    • Tracce totali: {len(valid_idx)}
    • Tracce pileup: {pu_true.sum()}
    • Frazione pileup: {pu_true.mean():.2%}
    
    Cinematica:
    • pT medio: {pt_true.mean():.2f} GeV
    • pT max: {pt_true.max():.2f} GeV
    • η range: [{eta.min():.2f}, {eta.max():.2f}]
    
    Performance Modello:
    • Loss ricostruzione: {losses.get('masked', 0):.4f}
    • Loss pileup: {losses.get('pu', 0):.4f}
    • Accuracy pileup: {pu_accuracy:.3f}
    • AUC pileup: {pu_auc:.3f}
    
    {'='*30}
    """
    ax6.text(0.1, 0.9, info_text, va='top', fontfamily='monospace', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    return outputs, losses

In [9]:
def main():
    # Configurazione
    config = get_model_config()
    config.update({
        'lr': 1e-4,
        'n_epochs': 10,
        'batch_size': 8,  # Dipende dalla memoria GPU
        'max_grad_norm': 1.0,
    })
    
    # Task weights per Stage 1 (Masked + PU)
    task_weights = {
        'masked': 0.35,  # 35% come dal piano
        'pu': 0.20,      # 20%
    }
    
    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Carica dati
    print("Loading dataset...")
    df = pd.read_parquet("atlas_tracks_sample.parquet")
    
    # Calcola statistiche per normalizzazione
    stats = compute_dataset_stats(df)  # Da implementare
    
    # Crea dataset
    full_dataset = AtlasTrackDataset(df, stats)
    
    # Split train/val
    train_size = int(0.9 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    # DataLoader
    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        collate_fn=collate_fn,  # Da implementare
        num_workers=4,
        pin_memory=True  # Importante per GPU
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=True
    )
    
    # Crea modello
    print("Creating model...")
    model = HierarchicalPerceiver(config)
    model = model.to(device)
    
    # Conta parametri
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {n_params:,}")
    
    # Trainer
    trainer = Trainer(model, device, config)
    
    # Training
    print("Starting training...")
    train_losses, val_losses = trainer.train(
        train_loader, 
        val_loader,
        n_epochs=config['n_epochs'],
        task_weights=task_weights,
        save_path='atlas_foundation_checkpoints'
    )
    
    print("Training complete!")
    
    # Plot losses
    plt.plot(train_losses, label='Train')
    plt.plot(val_losses, label='Validation')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    outputs, losses = visualize_event(model, val_dataset, event_idx=0, device=device)
    print("Loss calcolate:", losses)


if __name__ == "__main__":  
    main()

Using device: cuda
Loading dataset...
Dataset inizializzato con 1000 eventi
Features PF: ['pt', 'log_pt', 'eta', 'phi_x', 'phi_y', 'energy', 'd0', 'z0', 'charge']
Creating model...
Total parameters: 8,080,266
Starting training...

Epoch 1/10


Training:  30%|███       | 34/113 [00:04<00:10,  7.66it/s, loss=0.0996]


KeyboardInterrupt: 